In [59]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

In [60]:
from datasets import load_dataset

ds = load_dataset("seeeeiii/RICO-WidgetCaptioning")

In [61]:
ds

DatasetDict({
    train: Dataset({
        features: ['screenId', 'captions', 'view_hierarchy', 'bbox', 'file_name', 'file_name_semantic', 'semantic_annotations', 'app_package_name', 'play_store_name', 'category', 'average_rating', 'number_of_ratings', 'number_of_downloads', 'file_name_icon', 'image', 'image_icon', 'image_semantic'],
        num_rows: 41221
    })
    val: Dataset({
        features: ['screenId', 'captions', 'view_hierarchy', 'bbox', 'file_name', 'file_name_semantic', 'semantic_annotations', 'app_package_name', 'play_store_name', 'category', 'average_rating', 'number_of_ratings', 'number_of_downloads', 'file_name_icon', 'image', 'image_icon', 'image_semantic'],
        num_rows: 3483
    })
    test: Dataset({
        features: ['screenId', 'captions', 'view_hierarchy', 'bbox', 'file_name', 'file_name_semantic', 'semantic_annotations', 'app_package_name', 'play_store_name', 'category', 'average_rating', 'number_of_ratings', 'number_of_downloads', 'file_name_icon', 'ima

In [62]:
train_ds = ds['train']
valid_ds = ds['val']
test_ds = ds['test']

In [63]:
del ds

In [64]:
def simplify_ds(ds):
  compact = ds.select_columns(
      ["screenId", "captions", "category", "app_package_name"]
  ).rename_columns({
      "screenId": "id",
      "app_package_name": "package_name",
  })
  return compact

In [65]:
categories = train_ds.unique("category")

print("Number of categories:", len(categories))
print(sorted(categories))

Number of categories: 28
['000 - 1', 'Art & Design', 'Auto & Vehicles', 'Beauty', 'Books & Reference', 'Business', 'Comics', 'Communication', 'Dating', 'Education', 'Entertainment', 'Events', 'Finance', 'Food & Drink', 'Health & Fitness', 'House & Home', 'Lifestyle', 'Maps & Navigation', 'Medical', 'Music & Audio', 'News & Magazines', 'Parenting', 'Shopping', 'Social', 'Sports', 'Travel & Local', 'Video Players & Editors', 'Weather']


In [66]:
train_ds = simplify_ds(train_ds)
valid_ds = simplify_ds(valid_ds)
test_ds = simplify_ds(test_ds)

In [67]:
def split_ds(ds):
  return ds.map(
    lambda row: {
      "text": "|".join(row['captions']),
      "label": row['category']
    }
  )

In [68]:
train_ds = split_ds(train_ds)
valid_ds = split_ds(valid_ds)
test_ds = split_ds(test_ds)
train_ds

Dataset({
    features: ['id', 'captions', 'category', 'package_name', 'text', 'label'],
    num_rows: 41221
})

In [69]:
X_train, y_train = train_ds["text"], train_ds["label"]
X_valid, y_valid = valid_ds["text"], valid_ds["label"]
X_test, y_test = test_ds["text"], test_ds["label"]


In [70]:
len(X_train), len(y_train)

(41221, 41221)

In [71]:
X_train[0], y_train[0]

('get more information|go to options|more information', 'Travel & Local')

In [72]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_valid_tfidf = vectorizer.transform(X_valid)

In [73]:
print(X_train_tfidf.shape)
print(vectorizer.get_feature_names_out()[:20])

(41221, 23979)
['00' '000' '03' '10' '10 day' '10 min' '10 minutes' '10 seconds'
 '10 select' '100' '1017' '1017 vs' '11' '11 select' '11th' '11th day'
 '12' '12 hour' '12h' '12h clock']


In [74]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42,
)

In [75]:
model.fit(X_train_tfidf, y_train)

valid_predictions = model.predict(X_valid_tfidf)

print("Validation accuracy:", accuracy_score(y_valid, valid_predictions))
print(classification_report(y_valid, valid_predictions))

Validation accuracy: 0.2147573930519667
                         precision    recall  f1-score   support

           Art & Design       1.00      0.07      0.12       123
        Auto & Vehicles       0.00      0.00      0.00        16
                 Beauty       0.00      0.00      0.00        13
      Books & Reference       0.10      0.19      0.13       164
               Business       0.20      0.12      0.15        80
                 Comics       0.10      0.02      0.03        59
          Communication       0.35      0.33      0.34       264
                 Dating       0.30      0.05      0.09        55
              Education       0.10      0.24      0.14       108
          Entertainment       0.13      0.41      0.20       153
                 Events       0.00      0.00      0.00        27
                Finance       0.26      0.18      0.21       108
           Food & Drink       0.72      0.15      0.25       153
       Health & Fitness       0.17      0.27     

c:\Users\Mark\screenshot-classifier\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Mark\screenshot-classifier\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Mark\screenshot-classifier\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

In [76]:
for text, actual, predicted in zip(X_valid[:10], y_valid[:10], valid_predictions[:10]):
    print(f"Text:      {text}")
    print(f"Actual:    {actual}")
    print(f"Predicted: {predicted}")
    print()

Text:      look for|search|search
Actual:    Maps & Navigation
Predicted: Lifestyle

Text:      contacts|go to friends
Actual:    Maps & Navigation
Predicted: Communication

Text:      browse worldwide|world search
Actual:    Maps & Navigation
Predicted: Education

Text:      go to settings|settings
Actual:    Maps & Navigation
Predicted: Communication

Text:      enter e-mail address|name|type email address
Actual:    Shopping
Predicted: Sports

Text:      enter password|enter password|insert password
Actual:    Shopping
Predicted: Finance

Text:      get suggestions about this feature|help|info on saving email
Actual:    Shopping
Predicted: Health & Fitness

Text:      add email to autofill|save email
Actual:    Shopping
Predicted: Communication

Text:      search by phone number or name|send message
Actual:    Communication
Predicted: Communication

Text:      favorite chicken stuffed kulcha|favorite recipe|like button
Actual:    Food & Drink
Predicted: Food & Drink



In [77]:
from collections import Counter

train_counts = Counter(y_train)

for category, count in train_counts.most_common():
    print(f"{category}: {count}")

Entertainment: 3699
Health & Fitness: 2957
Education: 2849
Lifestyle: 2764
Social: 2586
Music & Audio: 2419
Shopping: 2331
Communication: 2245
Books & Reference: 2195
News & Magazines: 1899
Travel & Local: 1809
Sports: 1763
Finance: 1447
Weather: 1402
Medical: 1152
Business: 1111
Video Players & Editors: 996
Maps & Navigation: 950
Food & Drink: 917
Comics: 794
Parenting: 686
Dating: 681
Auto & Vehicles: 487
Beauty: 356
Art & Design: 335
House & Home: 318
Events: 70
000 - 1: 3


In [78]:
most_common_category, count = train_counts.most_common(1)[0]
baseline_accuracy = count / len(y_train)

print(most_common_category)
print(f"Baseline accuracy: {baseline_accuracy:.2%}")

Entertainment
Baseline accuracy: 8.97%


In [79]:
print(set(y_valid) - set(y_train))

set()


In [80]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42,
    )),
])

In [81]:
pipeline.fit(X_train, y_train)

valid_predictions = pipeline.predict(X_valid)

print(f"Validation accuracy: {accuracy_score(y_valid, valid_predictions):.2%}")
print(classification_report(y_valid, valid_predictions))

Validation accuracy: 21.48%
                         precision    recall  f1-score   support

           Art & Design       1.00      0.07      0.12       123
        Auto & Vehicles       0.00      0.00      0.00        16
                 Beauty       0.00      0.00      0.00        13
      Books & Reference       0.10      0.19      0.13       164
               Business       0.20      0.12      0.15        80
                 Comics       0.10      0.02      0.03        59
          Communication       0.35      0.33      0.34       264
                 Dating       0.30      0.05      0.09        55
              Education       0.10      0.24      0.14       108
          Entertainment       0.13      0.41      0.20       153
                 Events       0.00      0.00      0.00        27
                Finance       0.26      0.18      0.21       108
           Food & Drink       0.72      0.15      0.25       153
       Health & Fitness       0.17      0.27      0.21       

c:\Users\Mark\screenshot-classifier\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Mark\screenshot-classifier\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Mark\screenshot-classifier\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

In [82]:
for text, actual, predicted in zip(X_valid[:10], y_valid[:10], valid_predictions[:10]):
    print(f"Caption:   {text}")
    print(f"Actual:    {actual}")
    print(f"Predicted: {predicted}\n")

Caption:   look for|search|search
Actual:    Maps & Navigation
Predicted: Lifestyle

Caption:   contacts|go to friends
Actual:    Maps & Navigation
Predicted: Communication

Caption:   browse worldwide|world search
Actual:    Maps & Navigation
Predicted: Education

Caption:   go to settings|settings
Actual:    Maps & Navigation
Predicted: Communication

Caption:   enter e-mail address|name|type email address
Actual:    Shopping
Predicted: Sports

Caption:   enter password|enter password|insert password
Actual:    Shopping
Predicted: Finance

Caption:   get suggestions about this feature|help|info on saving email
Actual:    Shopping
Predicted: Health & Fitness

Caption:   add email to autofill|save email
Actual:    Shopping
Predicted: Communication

Caption:   search by phone number or name|send message
Actual:    Communication
Predicted: Communication

Caption:   favorite chicken stuffed kulcha|favorite recipe|like button
Actual:    Food & Drink
Predicted: Food & Drink



In [85]:
from pathlib import Path
import json

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import StringTensorType

export_dir = Path("models")
export_dir.mkdir(exist_ok=True)

tfidf_options = {
    "tfidf": {
        "separators": [
            " ", "[.]", "\\?", ",", ";", ":", "\\!",
            "\\(", "\\)", "\\[", "\\]", "\\|", "\\-", "\n",
            "\"", "'",
        ]
    },
    "classifier": {
        "zipmap": False,
        "nocl": True,
    },
}

onnx_model = convert_sklearn(
    pipeline,
    initial_types=[("text", StringTensorType([None, 1]))],
    options=tfidf_options,
    target_opset=17,
)

onnx_path = export_dir / "category_classifier.onnx"
onnx_path.write_bytes(onnx_model.SerializeToString())

labels_path = export_dir / "labels.json"
labels_path.write_text(
    json.dumps(pipeline.named_steps["classifier"].classes_.tolist()),
    encoding="utf-8",
)

print(onnx_path.resolve())
print(labels_path.resolve())

C:\Users\Mark\screenshot-classifier\src\notebooks\models\category_classifier.onnx
C:\Users\Mark\screenshot-classifier\src\notebooks\models\labels.json


In [86]:
import numpy as np
import onnxruntime as ort

sample_texts = X_valid[:10]

session = ort.InferenceSession(
    "models/category_classifier.onnx",
    providers=["CPUExecutionProvider"],
)

onnx_inputs = np.asarray(sample_texts, dtype=object).reshape(-1, 1)
outputs = session.run(None, {"text": onnx_inputs})

print([output.shape for output in outputs if hasattr(output, "shape")])
print("Python:", pipeline.predict(sample_texts))

[(10,), (10, 28)]
Python: ['Lifestyle' 'Communication' 'Education' 'Communication' 'Sports'
 'Finance' 'Health & Fitness' 'Communication' 'Communication'
 'Food & Drink']


In [ ]:
print([item.name for item in session.get_inputs()])
print([item.name for item in session.get_outputs()])

['text']
['label', 'probabilities']


: 